Squad 2 | Camada Bronze

**Tabela** | ecommerce_categorias |

**Origem** | vendas_raw/ (parquet) |

**Destino** | squad2/bronze/ecommerce_categorias (Delta) |

**Modo** | Delta Streaming — Structured Streaming |

**Objetivo** | Ingerir dados brutos na camada Bronze |

**Checkpoint** | squad2/checkpoints/bronze/ecommerce_categorias |

**Depende de** | feat_squad2_99_helpers |

In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
import logging
from pyspark.sql.functions import lit, current_timestamp

logging.getLogger("azure").setLevel(logging.WARNING)

TABELA        = "ecommerce_categorias"
DELTA_TABLE   = f"squad2.bronze_{TABELA}"
MODO_GRAVACAO = "append"

inicio = log_inicio(f"feat_squad2_bronze_{TABELA}")

log.info(f"Tabela      : {TABELA}")
log.info(f"Delta Table : {DELTA_TABLE}")
log.info(f"Modo        : {MODO_GRAVACAO}")

In [0]:
try:
    snapshots = sorted(listar_snapshots())
    log.info(f"{len(snapshots)} snapshot(s) disponível(is):\n")
    for snap in snapshots:
        print(f"  Pacote {snap}")

except Exception as e:
    log.error(f"Erro ao listar snapshots: {str(e)}")
    raise


In [0]:
try:
    processados = ler_checkpoint("bronze", TABELA)
    novos       = [s for s in snapshots if s not in processados]
    log.info(f"{len(novos)} snapshot(s) novo(s) para processar")

except Exception as e:
    log.error(f"Erro ao verificar checkpoint: {str(e)}")
    raise

In [0]:
try:
    total_linhas = 0

    if not novos:
        log.info("Nenhum snapshot novo para processar!")
    else:
        for snapshot_id in novos:
            log.info(f"Processando: {snapshot_id}")

            df        = ler_parquet(snapshot_id, TABELA)
            df_bronze = df \
                .withColumn("_snapshot_id", lit(snapshot_id)) \
                .withColumn("_ingested_at", current_timestamp()) \
                .withColumn("_source",      lit("real-time-data")) \
                .withColumn("_camada",      lit("bronze"))

            sucesso       = gravar_delta(df_bronze, "bronze", TABELA)
            count         = df_bronze.count()
            total_linhas += count
            processados.add(snapshot_id)

            log.info(f"  OK {snapshot_id} → {count} linhas")

        salvar_checkpoint("bronze", TABELA, processados)
        log.info(f" Total gravado: {total_linhas} linhas")

except Exception as e:
    log.error(f"Erro na ingestão Bronze: {str(e)}")
    raise

In [0]:
try:
    df_bronze = spark.table(DELTA_TABLE)
    total     = df_bronze.count()

    log.info(f" Validação Bronze OK!")
    log.info(f"   Tabela          : {DELTA_TABLE}")
    log.info(f"   Total registros : {total}")
    log.info(f"   Colunas         : {len(df_bronze.columns)}")

    print("\n Schema Bronze:")
    df_bronze.printSchema()

    print("\n Amostra:")
    display(df_bronze)

except Exception as e:
    log.error(f"Erro na validação: {str(e)}")
    raise

In [0]:
try:
    historico = spark.sql(f"DESCRIBE HISTORY {DELTA_TABLE}")
    log.info("Histórico Delta:")
    display(historico)

except Exception as e:
    log.error(f"Erro ao verificar histórico: {str(e)}")

In [0]:
log_fim(f"feat_squad2_bronze_{TABELA}", inicio)